In [42]:
from nba_api.stats.library.parameters import SeasonAll
from nba_api.stats.endpoints import playergamelog

# Get ALL of LeBron's games across his ENTIRE career in ONE call!
gamelog = playergamelog.PlayerGameLog(
    player_id='2544',
    season=SeasonAll.all
)
df = gamelog.get_data_frames()[0]
df

,SEASON_ID,Player_ID,Game_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,22025,2544,0022500078,"Nov 28, 2025",LAL vs. DAL,W,34,5,13,0.385,...,4,5,7,1,0,2,1,13,13,1
1,22025,2544,0022500059,"Nov 25, 2025",LAL vs. LAC,W,32,9,15,0.600,...,5,6,6,1,1,3,3,25,18,1
2,22025,2544,0022500282,"Nov 23, 2025",LAL @ UTA,W,34,8,18,0.444,...,6,6,8,1,0,2,2,17,-14,1
3,22025,2544,0022500253,"Nov 18, 2025",LAL vs. UTA,W,30,4,7,0.571,...,2,3,12,1,0,1,0,11,1,1
4,22024,2544,0022401185,"Apr 11, 2025",LAL vs. HOU,W,22,6,11,0.545,...,4,4,8,1,0,1,1,14,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1561,22003,2544,0020300068,"Nov 07, 2003",CLE @ IND,L,44,8,18,0.444,...,5,5,3,0,0,7,2,23,-7,0
1562,22003,2544,0020300057,"Nov 05, 2003",CLE vs. DEN,L,41,3,11,0.273,...,9,11,7,2,3,2,1,7,-3,0
1563,22003,2544,0020300038,"Nov 01, 2003",CLE @ POR,L,39,3,12,0.250,...,4,4,6,2,0,2,3,8,-21,0
1564,22003,2544,0020300018,"Oct 30, 2003",CLE @ PHX,L,41,8,17,0.471,...,10,12,8,1,0,7,1,21,-3,0


In [33]:
from nba_api.stats.endpoints import playercareerstats
from sqlalchemy import create_engine, text
from nba_api.stats.endpoints import leaguedashplayerstats
from nba_api.stats.endpoints import playerestimatedmetrics
from nba_api.stats.endpoints import playerdashboardbyyearoveryear
import pandas as pd


#get metric data 
metrics = playerestimatedmetrics.PlayerEstimatedMetrics()
metrics_df = metrics.get_data_frames()[0]
metrics_df.set_index('PLAYER_ID')
metrics_df

,PLAYER_ID,PLAYER_NAME,GP,W,L,W_PCT,MIN,E_OFF_RATING,E_DEF_RATING,E_NET_RATING,...,E_OFF_RATING_RANK,E_DEF_RATING_RANK,E_NET_RATING_RANK,E_AST_RATIO_RANK,E_OREB_PCT_RANK,E_DREB_PCT_RANK,E_REB_PCT_RANK,E_TOV_PCT_RANK,E_USG_PCT_RANK,E_PACE_RANK
0,1630230,Naji Marshall,21,6,15,0.286,26.9,103.1,109.7,-6.7,...,421,166,336,221,304,152,199,56,248,100
1,201566,Russell Westbrook,21,5,16,0.238,28.2,108.6,120.3,-11.8,...,329,424,398,48,157,61,96,56,88,123
2,1641711,Gradey Dick,21,14,7,0.667,16.2,116.3,103.1,13.1,...,130,57,57,405,339,333,353,56,281,123
3,1630700,Dyson Daniels,21,13,8,0.619,33.7,117.3,111.9,5.4,...,106,222,140,28,127,227,169,56,322,134
4,1642907,Cedric Coward,21,9,12,0.429,27.1,111.9,110.3,1.6,...,247,183,205,219,183,136,145,56,192,142
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
483,1641725,Trey Alexander,1,0,1,0.000,21.2,110.2,115.3,-5.1,...,285,325,310,11,449,373,440,56,444,462
484,1642942,Jahmai Mashack,1,1,0,1.000,6.4,93.2,86.2,7.0,...,468,11,116,456,449,2,68,1,45,465
485,1642950,Lachlan Olbrich,1,0,1,0.000,11.0,121.0,57.8,63.2,...,51,3,4,456,449,52,74,476,281,478
486,1642935,Chucky Hepburn,1,0,1,0.000,4.9,44.4,85.8,-41.4,...,488,10,478,456,449,219,375,1,69,484


In [31]:
from sqlalchemy import create_engine
from requests.exceptions import ReadTimeout
import random
import time 
import pandas as pd
import json

from nba_api.stats.static import players
from nba_api.stats.endpoints import playergamelog

#start by creating an engine to add each dataframe to the sql database
from dotenv import load_dotenv
import os
from nba_api.stats.endpoints import teamyearbyyearstats, commonteamroster
from nba_api.stats.static import teams
import time
import random

load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)

#filter all players for active players
#active_players = [p for p in players.get_players() if p['is_active']]
# Get all teams
print("Fetching all NBA teams...")
nba_teams = teams.get_teams()
print(f"Found {len(nba_teams)} teams in total")

all_players = []
team_count = 0
active_teams = 0

# for team in nba_teams:
#     team_id = team['id']
#     team_name = team['full_name']
#     team_count += 1
    
    # print(f"[{team_count}/{len(nba_teams)}] Processing team: {team_name} (ID: {team_id})")
    # print(team)
    
print(f"  Fetching season history for {team_name}...")
team_seasons = teamyearbyyearstats.TeamYearByYearStats(team_id=nba_teams[0]['id'])
seasons_df = team_seasons.get_data_frames()[0]

#lets try and get the 15 most recent seasons
#for i in range(len(my_list) - 1, len(my_list) - 11, -1):

seasons = []
for i in range(len(seasons_df) - 1, len(seasons_df) - 16, -1):
    season = seasons_df.iloc[i]['YEAR']
    seasons.append(season)
print(seasons)
#     print(seasons_df.iloc[i])
# print(seasons_df)

Fetching all NBA teams...
Found 30 teams in total
  Fetching season history for Charlotte Hornets...
['2025-26', '2024-25', '2023-24', '2022-23', '2021-22', '2020-21', '2019-20', '2018-19', '2017-18', '2016-17', '2015-16', '2014-15', '2013-14', '2012-13', '2011-12']


In [ ]:
from requests.exceptions import ReadTimeout
from sqlalchemy import create_engine, inspect
from dotenv import load_dotenv
import pandas as pd
import random
import time 
import json
import os

from nba_api.stats.library.parameters import SeasonAll
from nba_api.stats.static import players, teams
from nba_api.stats.endpoints import (
    boxscoreadvancedv2,
    playergamelog,
    teamyearbyyearstats,
    commonteamroster
)

#---------------------------------------------------------------------------------
#define helper functions
def get_processed_players(engine, table_name='player_game_stats_temp'):
    """Get set of player IDs we've already processed"""
    inspector = inspect(engine)
    if table_name in inspector.get_table_names():
        query = f"SELECT DISTINCT PLAYER_ID FROM {table_name}"
        processed_df = pd.read_sql(query, engine)
        return set(processed_df['PLAYER_ID'].tolist())
    return set()

# def get_processed_games(engine, table_name='player_game_stats'):
#     """Get set of game IDs we've already fetched advanced stats for"""
#     inspector = inspect(engine)
#     if table_name in inspector.get_table_names():
#         # Check if the advanced stats columns exist (indicating we've processed this game)
#         query = f"SELECT DISTINCT GAME_ID FROM {table_name} WHERE E_OFF_RATING IS NOT NULL"
#         processed_df = pd.read_sql(query, engine)
#         return set(processed_df['GAME_ID'].tolist())
#     return set()

# def chunks(lst, n):
#     """Split list into chunks of size n"""
#     for i in range(0, len(lst), n):
#         yield lst[i:i + n]

# from nba_api.stats.static import players, teams
# from nba_api.stats.endpoints import playergamelog
# from nba_api.stats.library.parameters import SeasonAll

# from nba_api.stats.endpoints import teamyearbyyearstats, commonteamroster
# from nba_api.stats.static import teams

#---------------------------------------------------------------------------
#Variable setup 
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

seasons = ['2023-24', '2022-23', '2021-22', '2020-21', '2019-20',
           '2018-19', '2017-18', '2016-17', '2015-16', '2014-15',
           '2013-14', '2012-13', '2011-12', '2010-11', '2009-10']

nba_teams = teams.get_teams()
print(f"Found {len(nba_teams)} teams in total")

#----------------------------------------------------------------------------
#get players based on team rosters in each year we are targeting 

all_players = []
team_count = 0
seen_player_ids = set()

for team in nba_teams:
    team_id = team['id']
    team_name = team['full_name']
    team_count += 1
    
    print(f"[{team_count}/{len(nba_teams)}] Processing team: {team_name} (ID: {team_id})")
    
    try:
        # Check if the team was active that season
        # print(f"  Fetching season history for {team_name}...")
        # team_seasons = teamyearbyyearstats.TeamYearByYearStats(team_id=team_id)
        # seasons_df = team_seasons.get_data_frames()[0]
        
        # # Add a sleep to avoid rate limiting
        # sleep_time = random.uniform(1.5, 3.0)
        # print(f"  Sleeping for {sleep_time:.2f} seconds...")
        # time.sleep(sleep_time)
        
        for season in seasons:
            if season in seasons_df['YEAR'].values:
                
                # Add a sleep to avoid rate limiting
                sleep_time = random.uniform(1.5, 3.0)
                print(f"  Sleeping for {sleep_time:.2f} seconds...")
                time.sleep(sleep_time)
                
                # Get team roster for that season
                print(f"  Fetching {season} roster for {team_name}...")
                roster = commonteamroster.CommonTeamRoster(
                    team_id=team_id,
                    season=season
                )
                roster_df = roster.get_data_frames()[0]

                # Filter out players we've already seen
                # Convert to dict records
                # Update the set of seen player IDs
                new_players_df = roster_df[~roster_df['PLAYER_ID'].isin(seen_player_ids)]
                new_players = new_players_df.to_dict('records')
                seen_player_ids.update(new_players_df['PLAYER_ID'].tolist())
                
                player_count = len(new_players)
                all_players.extend(new_players)
                print(f"  ✓ Added {player_count} players from {team_name}")
                
            else:
                print(f"  ✗ {team_name} was NOT active in {season} season")
        
    except Exception as e:
        print(f"  ⚠ Error processing {team_name}: {str(e)}")
    
    # Add a separator for readability
    print("-" * 50)
    
print("\nSummary:")
print(f"Processed {team_count} total teams")
print(f"Collected data for {len(all_players)} players")
print(len(all_players))

#------------------------------------------------------------------------------
#gather player games

# Get already processed players
print("Checking for existing progress...")
processed_player_ids = get_processed_players(engine, 'player_game_stats_temp')
print(f"Found {len(processed_player_ids)} players already processed")

# Filter out already-processed players
players_to_process = [p for p in all_players if p['PLAYER_ID'] not in processed_player_ids]
print(f"Remaining players to process: {len(players_to_process)}")

#setup dataframe
player_stats_df = pd.DataFrame()
i = 1

for player in players_to_process:
    while True:
        try:
            print(f"Getting stats for {player['PLAYER']}")
            #get player dict and ID
            PID = player['PLAYER_ID']
            
            #retreive player game stats
            game_log = playergamelog.PlayerGameLog(
                player_id=PID,
                season=SeasonAll.all
            )
            df = game_log.get_data_frames()[0]

            if not df.empty:
                df = df[df['SEASON_ID'].str[-7:].isin(seasons)]
                if not df.empty:
                    player_stats_df = pd.concat([player_stats_df, df], ignore_index=True)
                    print(f"Data retreived for {player['PLAYER']}  - {season} ({len(df)} games)")

                    # SAVE PROGRESS every 50 players
                    if i % 50 == 0:
                        print(f"\n  💾 Saving checkpoint at player {i}...")
                        player_stats_df.to_sql(
                            'player_game_stats_temp',
                            engine,
                            if_exists='append',
                            index=False
                        )
                        player_stats_df = pd.DataFrame()  # Clear memory
                        print("  ✓ Checkpoint saved\n")
                else:
                    print(f"No games in target seasons for {player['PLAYER']}")
            else:
                print(f"No games found for {player['PLAYER']} in {season}")
        
            if i % 40 == 0:
                time.sleep(round(random.uniform(60, 120), 1))
                print()
                print("Long Sleep!")
                print()
                
            else:
                time.sleep(round(random.uniform(3, 4), 1))
            i += 1

            #exit loop
            break 
            
        except (ReadTimeout, json.decoder.JSONDecodeError, Exception) as e:
            print(f"Error for {player['PLAYER']}: {e} - retrying after 60 seconds")
            time.sleep(180)
            continue 

# Save any remaining data
if not player_stats_df.empty:
    player_stats_df.to_sql('player_game_stats_temp', engine, if_exists='append', index=False)

# Load all collected player stats
print("\nLoading all collected player stats...")
all_player_stats = pd.read_sql('SELECT * FROM player_game_stats_temp', engine)

#-------------------------------------------------------------------------------------
#Get UNIQUE game IDs from collected data

# Extract ALL unique game IDs from the complete dataset
# all_game_ids = set(all_player_stats['GAME_ID'].unique())

# print("\nChecking which games already have advanced stats...")
# processed_game_ids = get_processed_games(engine, 'player_game_stats')
# games_to_process = list(all_game_ids - processed_game_ids)

# print(f"  Total games needed: {len(all_game_ids)}")
# print(f"  Already processed: {len(processed_game_ids)}")
# print(f"  Remaining to fetch: {len(games_to_process)}")

# if len(games_to_process) == 0:
#     print("\n✅ All games already have advanced stats! Nothing to do.")
#     exit()

# #-------------------------------------------------------------------------------------
# # define functions for batching game data and saving progress
# print("\nStep 4: Fetching advanced stats in batches...")

# game_batches = list(chunks(games_to_process, 100))
# print(f"Processing {len(games_to_process)} games in {len(game_batches)} batches")

# #-------------------------------------------------------------------------------------
# # Fetch advanced stats in batches
# j = 1

# for batch_num, batch in enumerate(game_batches, 1):
#     print(f"\n--- Batch {batch_num}/{len(game_batches)} ---")
#     batch_advanced_stats = pd.DataFrame()
    
#     for game_id in batch:
#         while True:
#             try:
#                 print(f"Fetching advanced stats for game {j}/{len(games_to_process)}: {game_id}")
                
#                 advanced = boxscoreadvancedv2.BoxScoreAdvancedV2(game_id=game_id)
#                 advanced_df = advanced.get_data_frames()[0]  # Player stats
#                 batch_advanced_stats = pd.concat([batch_advanced_stats, advanced_df], ignore_index=True)
                
#                 # Rate limiting
#                 if j % 100 == 0:
#                     time.sleep(random.uniform(60, 120))
#                 else:
#                     time.sleep(random.uniform(1, 2))
                
#                 j += 1
#                 break
                
#             except Exception as e:
#                 print(f"Error fetching game {game_id}: {e} - retrying")
#                 time.sleep(180)
#                 continue


#     # KEY CHANGE: Merge this batch and save to final table incrementally
#     print(f"\nMerging and saving batch {batch_num}...")
    
#     # Get the player stats for games in this batch
#     batch_game_ids = batch_advanced_stats['GAME_ID'].unique()
#     batch_player_stats = all_player_stats[all_player_stats['GAME_ID'].isin(batch_game_ids)]
    
#     # Merge basic and advanced for this batch
#     batch_combined = batch_player_stats.merge(
#         batch_advanced_stats,
#         on=['GAME_ID', 'PLAYER_ID'],
#         how='left',
#         suffixes=('', '_adv')
#     )
    
#     # Save to database (append after first batch)
#     batch_combined.to_sql(
#         'player_game_stats',
#         engine,
#         if_exists='append',
#         index=False
#     )
    
#     print(f"✓ Batch {batch_num} saved ({len(batch_combined)} records)")
#     print(f"  Progress: {j-1}/{len(games_to_process)} games complete")

print("\n" + "="*60)
print("✅ Complete! All data saved to 'player_game_stats'")
print("="*60)
# Step 4: Merge locally (instant, no API calls!)
# print("\nStep 4: Merging basic and advanced stats...")
# combined_df = player_stats_df.merge(
#     all_advanced_stats,
#     on=['GAME_ID', 'PLAYER_ID'],
#     how='left',
#     suffixes=('', '_adv')
# )

# print(f"\nFinal dataset: {len(combined_df)} rows")

# Step 5: Save to database
# filename = "player_complete_game_stats"
# combined_df.to_sql(filename, engine, if_exists='replace', index=False)
# print(f"✅ Saved to database!")
# filename = f"playerStats"
# player_stats_df.to_sql(filename, engine, if_exists='replace', index=False)
# print(f"Finished gathering all data for all players for all games in seasons {seasons[0]} to {seasons[-1]}")